# Pose Detection with AlphaPhose

This notebook uses an open source project [MVIG-SJTU/AlphaPose](https://github.com/MVIG-SJTU/AlphaPose) to detect/track multi person poses on a given youtube video.

For other deep-learning Colab notebooks, visit [tugstugi/dl-colab-notebooks](https://github.com/tugstugi/dl-colab-notebooks).


## Install AlphaPose

In [ ]:
import os
from os.path import exists, join, basename, splitext

git_repo_url = 'https://github.com/MVIG-SJTU/AlphaPose.git'
project_name = splitext(basename(git_repo_url))[0]
if not exists(project_name):
  # clone and install dependencies
  !git clone -q --depth 1 $git_repo_url
  !cd $project_name && pip install -q -r requirements.txt
  !pip install -q gdown youtube-dl visdom

import sys
sys.path.append(project_name)
import time
import matplotlib
import matplotlib.pylab as plt
plt.rcParams["axes.grid"] = False

from IPython.display import YouTubeVideo

## Download pretrained models

In [ ]:
import gdown

# Download halpe_26 ResNet50 256x192 pretrained model
pretrained_model_path = join(project_name, 'pretrained_models/halpe26_fast_res50_256x192.pth')
os.makedirs(join(project_name, 'pretrained_models'), exist_ok=True)
if not exists(pretrained_model_path):
  gdown.download(
    'https://drive.google.com/uc?id=1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb',
    pretrained_model_path,
    quiet=False
  )

# Download YOLO detector weights
yolo_pretrained_model_path = join(project_name, 'detector/yolo/data/yolov3-spp.weights')
os.makedirs(join(project_name, 'detector/yolo/data'), exist_ok=True)
if not exists(yolo_pretrained_model_path):
  gdown.download(
    'https://drive.google.com/uc?id=1D47msNOOiJKvPOXlnpyzdKA3k6E97NTC',
    yolo_pretrained_model_path,
    quiet=False
  )

## Detect poses on a test video

We are going to detect poses on the following youtube video:

In [ ]:
YOUTUBE_ID = 'RXABo9hm8B8'


YouTubeVideo(YOUTUBE_ID)

Download the above youtube video, cut the first 5 seconds and do the pose detection on that 5 seconds:

In [ ]:
!rm -df youtube.mp4
# download the youtube video with the given ID
!youtube-dl -f 'bestvideo[ext=mp4]' --output "youtube.%(ext)s" https://www.youtube.com/watch?v=$YOUTUBE_ID
# cut the first 5 seconds
!ffmpeg -y -loglevel info -i youtube.mp4 -t 5 video.mp4
# run AlphaPose on these 5 seconds using halpe_26 config
!rm -rf AlphaPose_video.avi
!cd $project_name && python scripts/demo_inference.py \
  --cfg configs/halpe_26/resnet/256x192_res50_lr1e-3_1x.yaml \
  --checkpoint pretrained_models/halpe26_fast_res50_256x192.pth \
  --video ../video.mp4 \
  --outdir .. \
  --save_video
# convert the result into MP4
!ffmpeg -y -loglevel info -i AlphaPose_video.avi AlphaPose_video.mp4

Finally, visualize the result:

In [ ]:
def show_local_mp4_video(file_name, width=640, height=480):
  import io
  import base64
  from IPython.display import HTML
  video_encoded = base64.b64encode(io.open(file_name, 'rb').read())
  return HTML(data='''<video width="{0}" height="{1}" alt="test" controls>
                        <source src="data:video/mp4;base64,{2}" type="video/mp4" />
                      </video>'''.format(width, height, video_encoded.decode('ascii')))

show_local_mp4_video('AlphaPose_video.mp4', width=960, height=720)